In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/train.csv.zip
/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/sample_submission.csv.zip
/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/test_labels.csv.zip
/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/test.csv.zip


In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = '/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/'
train_df = pd.read_csv(DATA_PATH + 'train.csv.zip')

# 2. 印出基本資訊 (總共有幾筆資料、欄位型態)
print("數據資訊")
print(f"總評論筆數: {len(train_df)} 筆")
print(f"欄位名稱: {list(train_df.columns)}\n")

target_columns = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
print("各個標籤的比例")

stats = []
for col in target_columns:
    count = train_df[col].sum()
    percentage = (count / len(train_df)) * 100
    stats.append({'Label': col, 'Count': count, 'Percentage (%)': round(percentage, 2)})
    print(f"{col:<15} -> 惡意件數: {count:>6} 筆 | 佔總體: {percentage:>6.2f}%")


數據資訊
總評論筆數: 159571 筆
欄位名稱: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

各個標籤的比例
toxic           -> 惡意件數:  15294 筆 | 佔總體:   9.58%
severe_toxic    -> 惡意件數:   1595 筆 | 佔總體:   1.00%
obscene         -> 惡意件數:   8449 筆 | 佔總體:   5.29%
threat          -> 惡意件數:    478 筆 | 佔總體:   0.30%
insult          -> 惡意件數:   7877 筆 | 佔總體:   4.94%
identity_hate   -> 惡意件數:   1405 筆 | 佔總體:   0.88%


In [15]:
import zipfile
import pandas as pd
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier

INPUT_DIR = '/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/'

with zipfile.ZipFile(os.path.join(INPUT_DIR, 'train.csv.zip'), 'r') as zip_ref:
    zip_ref.extractall('/kaggle/working/')
with zipfile.ZipFile(os.path.join(INPUT_DIR, 'test.csv.zip'), 'r') as zip_ref:
    zip_ref.extractall('/kaggle/working/')

train_df = pd.read_csv('/kaggle/working/train.csv')
test_df = pd.read_csv('/kaggle/working/test.csv')

vectorizer = TfidfVectorizer(max_features=30000, stop_words='english')

X_train = vectorizer.fit_transform(train_df['comment_text'])
X_test = vectorizer.transform(test_df['comment_text'])

target_columns = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
y_train = train_df[target_columns]



In [ ]:
model = MultiOutputClassifier(LogisticRegression(C=2.0, max_iter=200, solver='liblinear'))
model.fit(X_train, y_train)

predictions = model.predict_proba(X_test)

submission = pd.DataFrame({'id': test_df['id']})
for i, col in enumerate(target_columns):
    submission[col] = predictions[i][:, 1]



submission.to_csv('submission_baseline.csv', index=False)

submission.head()

In [4]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.optim as optim
import numpy as np

MODEL_NAME = 'bert-base-uncased'
MAX_LEN = 128     
BATCH_SIZE = 32      
EPOCHS = 1           
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class ToxicDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        inputs = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )
        
        item = {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten()
        }
        
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
            
        return item

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = ToxicDataset(train_df['comment_text'].values, y_train.values, tokenizer, MAX_LEN)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

test_dataset = ToxicDataset(test_df['comment_text'].values, labels=None, tokenizer=tokenizer, max_len=MAX_LEN)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=6)
model.to(device)

optimizer = optim.AdamW(model.parameters(), lr=2e-5)
criterion = torch.nn.BCEWithLogitsLoss()

model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for step, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        
        loss = criterion(outputs.logits, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        if step % 100 == 0 and step > 0:
            print(f"Epoch {epoch+1} | Step {step}/{len(train_loader)} | Loss: {total_loss/step:.4f}")


model.eval()
bert_preds = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        probs = torch.sigmoid(outputs.logits).cpu().numpy()
        bert_preds.append(probs)


bert_preds = np.vstack(bert_preds)
submission_bert = pd.DataFrame({'id': test_df['id']})
for i, col in enumerate(target_columns):
    submission_bert[col] = bert_preds[:, i]

submission_bert.to_csv('submission_bert.csv', index=False)
submission_bert.head()

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1 | Step 100/4987 | Loss: 0.2496
Epoch 1 | Step 200/4987 | Loss: 0.1667
Epoch 1 | Step 300/4987 | Loss: 0.1333
Epoch 1 | Step 400/4987 | Loss: 0.1140
Epoch 1 | Step 500/4987 | Loss: 0.1015
Epoch 1 | Step 600/4987 | Loss: 0.0936
Epoch 1 | Step 700/4987 | Loss: 0.0870
Epoch 1 | Step 800/4987 | Loss: 0.0827
Epoch 1 | Step 900/4987 | Loss: 0.0793
Epoch 1 | Step 1000/4987 | Loss: 0.0765
Epoch 1 | Step 1100/4987 | Loss: 0.0742
Epoch 1 | Step 1200/4987 | Loss: 0.0717
Epoch 1 | Step 1300/4987 | Loss: 0.0696
Epoch 1 | Step 1400/4987 | Loss: 0.0678
Epoch 1 | Step 1500/4987 | Loss: 0.0663
Epoch 1 | Step 1600/4987 | Loss: 0.0651
Epoch 1 | Step 1700/4987 | Loss: 0.0641
Epoch 1 | Step 1800/4987 | Loss: 0.0630
Epoch 1 | Step 1900/4987 | Loss: 0.0618
Epoch 1 | Step 2000/4987 | Loss: 0.0607
Epoch 1 | Step 2100/4987 | Loss: 0.0598
Epoch 1 | Step 2200/4987 | Loss: 0.0589
Epoch 1 | Step 2300/4987 | Loss: 0.0580
Epoch 1 | Step 2400/4987 | Loss: 0.0573
Epoch 1 | Step 2500/4987 | Loss: 0.0567
Epoch 1 |

,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,00001cee341fdb12,0.997260,0.597612,0.987216,0.103551,0.956951,0.524453
1,0000247867823ef7,0.000988,0.000276,0.000498,0.000284,0.000412,0.000276
2,00013b17ad220c46,0.001260,0.000229,0.000612,0.000217,0.000404,0.000279
3,00017563c3f7919a,0.000801,0.000341,0.000495,0.000355,0.000415,0.000324
4,00017695ad8997eb,0.001451,0.000224,0.000552,0.000243,0.000411,0.000242


In [3]:
!pip install wandb

In [13]:
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.optim as optim
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import wandb

MODEL_NAME = 'microsoft/deberta-v3-small' 
MAX_LEN = 128     
BATCH_SIZE = 32     
EPOCHS = 1           
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

wandb.init(project="jigsaw-toxic-bert", name="deberta-v3-weighted", config={
    "model": MODEL_NAME,
    "lr": 1e-5,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "max_len": MAX_LEN,
    "notes": "DeBERTa-v3 with class weights"
})

X_train_text, X_val_text, y_train_labels, y_val_labels = train_test_split(
    train_df['comment_text'].values, 
    y_train.values, 
    test_size=0.2, 
    random_state=42
)

pos_counts = y_train_labels.sum(axis=0)
neg_counts = len(y_train_labels) - pos_counts
raw_weight = neg_counts / (pos_counts + 1e-5)
smoothed_weight = np.sqrt(raw_weight) 
pos_weight = torch.tensor(smoothed_weight, dtype=torch.float).to(device)

class ToxicDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        inputs = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )
        item = {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten()
        }
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

train_dataset = ToxicDataset(X_train_text, y_train_labels, tokenizer, MAX_LEN)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

val_dataset = ToxicDataset(X_val_text, y_val_labels, tokenizer, MAX_LEN)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

test_dataset = ToxicDataset(test_df['comment_text'].values, labels=None, tokenizer=tokenizer, max_len=MAX_LEN)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=6)
model = model.float()
model.to(device)

optimizer = optim.AdamW(model.parameters(), lr=1e-5)
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for step, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        
        loss = criterion(outputs.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        
        if step % 100 == 0 and step > 0:
            current_loss = total_loss / step
            print(f"Epoch {epoch+1} | Step {step}/{len(train_loader)} | Loss: {current_loss:.4f}")
            wandb.log({"train_loss": current_loss, "step": step})

    model.eval()
    val_preds = []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.sigmoid(outputs.logits).cpu().numpy()
            val_preds.append(probs)
            
    val_preds = np.vstack(val_preds)
    
    auc_scores = []
    for i, col in enumerate(target_columns):
        score = roc_auc_score(y_val_labels[:, i], val_preds[:, i])
        auc_scores.append(score)
        wandb.log({f"val_auc_{col}": score})
        
    mean_auc = np.mean(auc_scores)
    wandb.log({"mean_val_auc": mean_auc})

model.eval()
deberta_preds = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.sigmoid(outputs.logits).cpu().numpy()
        deberta_preds.append(probs)

deberta_preds = np.vstack(deberta_preds)

submission_deberta = pd.DataFrame({'id': test_df['id']})
for i, col in enumerate(target_columns):
    submission_deberta[col] = deberta_preds[:, i]

submission_deberta.to_csv('submission_deberta.csv', index=False)
wandb.finish()

mean_val_auc,▁
step,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
train_loss,█▆▅▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_auc_identity_hate,▁
val_auc_insult,▁
val_auc_obscene,▁
val_auc_severe_toxic,▁
val_auc_threat,▁
val_auc_toxic,▁
mean_val_auc,0.98687
step,3900


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight     

Epoch 1 | Step 100/3990 | Loss: 0.4932
Epoch 1 | Step 200/3990 | Loss: 0.3620
Epoch 1 | Step 300/3990 | Loss: 0.3056
Epoch 1 | Step 400/3990 | Loss: 0.2692
Epoch 1 | Step 500/3990 | Loss: 0.2442
Epoch 1 | Step 600/3990 | Loss: 0.2290
Epoch 1 | Step 700/3990 | Loss: 0.2211
Epoch 1 | Step 800/3990 | Loss: 0.2134
Epoch 1 | Step 900/3990 | Loss: 0.2065
Epoch 1 | Step 1000/3990 | Loss: 0.2018
Epoch 1 | Step 1100/3990 | Loss: 0.1949
Epoch 1 | Step 1200/3990 | Loss: 0.1899
Epoch 1 | Step 1300/3990 | Loss: 0.1863
Epoch 1 | Step 1400/3990 | Loss: 0.1833
Epoch 1 | Step 1500/3990 | Loss: 0.1804
Epoch 1 | Step 1600/3990 | Loss: 0.1785
Epoch 1 | Step 1700/3990 | Loss: 0.1770
Epoch 1 | Step 1800/3990 | Loss: 0.1759
Epoch 1 | Step 1900/3990 | Loss: 0.1733
Epoch 1 | Step 2000/3990 | Loss: 0.1704
Epoch 1 | Step 2100/3990 | Loss: 0.1684
Epoch 1 | Step 2200/3990 | Loss: 0.1671
Epoch 1 | Step 2300/3990 | Loss: 0.1658
Epoch 1 | Step 2400/3990 | Loss: 0.1653
Epoch 1 | Step 2500/3990 | Loss: 0.1637
Epoch 1 |

mean_val_auc,▁
step,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
train_loss,█▅▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_auc_identity_hate,▁
val_auc_insult,▁
val_auc_obscene,▁
val_auc_severe_toxic,▁
val_auc_threat,▁
val_auc_toxic,▁
mean_val_auc,0.98708
step,3900
